In [ ]:
import nbformat as nbf

nb = nbf.v4.new_notebook()
cells = []

def md(text):
    cells.append(nbf.v4.new_markdown_cell(text))

def code(text):
    cells.append(nbf.v4.new_code_cell(text))

# ---------------------------------------------------------------------------
md("""\
# Lab Question 2 — BBC News Classification with Word2Vec, GloVe & FastText

Using the BBC News Dataset, build and compare classification models based on
three embedding techniques: **Word2Vec**, **GloVe**, **FastText**.

**Pipeline:**
1. Build/load word vectors
2. Convert each news article into a single feature vector (average of its word vectors)
3. Train a classifier (SVM and Naive Bayes) on top of those features
4. Evaluate: Accuracy, Precision, Recall, F1-score, Confusion Matrix
5. Compare Word2Vec vs GloVe vs FastText
6. Save the best model with `joblib`
7. Reload the saved model and classify a new news article

**Data required (place in the same folder as this notebook):**
- `bbc-text.csv` — BBC News dataset with columns `category`, `text`
- `glove.2024.wikigiga.100d.txt` — pretrained GloVe vectors (2024 Wikipedia + Gigaword, 100d, uncased), from https://nlp.stanford.edu/projects/glove/

> Note: Naive Bayes normally expects non-negative features (that's why it's usually paired with bag-of-words/TF-IDF). Since Word2Vec/GloVe/FastText vectors contain negative values, we use `GaussianNB` instead of `MultinomialNB`, which correctly handles continuous, possibly-negative features.
""")

# ---------------------------------------------------------------------------
md("## 1. Setup & Imports")
code("""\
import os
import re
import numpy as np
import pandas as pd
import joblib

from gensim.models import Word2Vec, FastText

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
""")

# ---------------------------------------------------------------------------
md("## 2. Configuration")
code("""\
DATA_PATH = "bbc-text.csv"                    # BBC News dataset (columns: category, text)
GLOVE_PATH = "glove.2024.wikigiga.100d.txt"    # Pretrained GloVe vectors (2024 Wiki+Gigaword)
VECTOR_SIZE = 100
RANDOM_STATE = 42
MODEL_OUT_DIR = "saved_models"

os.makedirs(MODEL_OUT_DIR, exist_ok=True)
""")

# ---------------------------------------------------------------------------
md("## 3. Load & Clean Data")
code("""\
def load_data(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["text", "category"]).reset_index(drop=True)
    return df


def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text


def tokenize(text):
    return clean_text(text).split()
""")

code("""\
df = load_data(DATA_PATH)
df["tokens"] = df["text"].apply(tokenize)
print(df.shape)
df.head()
""")

# ---------------------------------------------------------------------------
md("""\
## 4. Embedding Builders
Word2Vec and FastText are trained from scratch on the training corpus.
GloVe vectors are pretrained and loaded from disk.
""")
code("""\
def train_word2vec(tokenized_corpus, vector_size=VECTOR_SIZE):
    model = Word2Vec(
        sentences=tokenized_corpus,
        vector_size=vector_size,
        window=5,
        min_count=2,
        workers=4,
        sg=1,  # skip-gram
        seed=RANDOM_STATE,
    )
    return model


def train_fasttext(tokenized_corpus, vector_size=VECTOR_SIZE):
    model = FastText(
        sentences=tokenized_corpus,
        vector_size=vector_size,
        window=5,
        min_count=2,
        workers=4,
        sg=1,
        seed=RANDOM_STATE,
    )
    return model


def load_glove(path, vector_size=VECTOR_SIZE):
    \"\"\"Loads GloVe vectors from a .txt file into a dict {word: vector}.\"\"\"
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"GloVe file not found at '{path}'. Download e.g. "
            f"glove.2024.wikigiga.100d.txt from "
            f"https://nlp.stanford.edu/projects/glove/ and place it here."
        )
    embeddings = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            vec = np.asarray(values[1:], dtype="float32")
            if len(vec) == vector_size:
                embeddings[word] = vec
    return embeddings
""")

# ---------------------------------------------------------------------------
md("## 5. Document Vectorization (average word vectors)")
code("""\
def document_vector_gensim(tokens, model, vector_size=VECTOR_SIZE):
    \"\"\"Average word vectors for a gensim model (Word2Vec / FastText).\"\"\"
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)


def document_vector_glove(tokens, glove_dict, vector_size=VECTOR_SIZE):
    vectors = [glove_dict[w] for w in tokens if w in glove_dict]
    if not vectors:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)


def build_feature_matrix(tokenized_docs, embed_type, model_or_dict):
    if embed_type in ("word2vec", "fasttext"):
        X = np.array(
            [document_vector_gensim(doc, model_or_dict) for doc in tokenized_docs]
        )
    elif embed_type == "glove":
        X = np.array(
            [document_vector_glove(doc, model_or_dict) for doc in tokenized_docs]
        )
    else:
        raise ValueError("Unknown embedding type")
    return X
""")

# ---------------------------------------------------------------------------
md("## 6. Train + Evaluate Classifier Helpers")
code("""\
def evaluate(y_true, y_pred, label_names, tag):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\\n=== {tag} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print("Confusion Matrix:")
    print(pd.DataFrame(cm, index=label_names, columns=label_names))
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "confusion_matrix": cm}


def train_and_evaluate(X_train, X_test, y_train, y_test, label_names, tag):
    results = {}

    # --- SVM ---
    svm_clf = SVC(kernel="linear", random_state=RANDOM_STATE)
    svm_clf.fit(X_train, y_train)
    y_pred_svm = svm_clf.predict(X_test)
    results["SVM"] = evaluate(y_test, y_pred_svm, label_names, f"{tag} - SVM")

    # --- Naive Bayes (GaussianNB, since embedding vectors are continuous
    #     and can be negative -> MultinomialNB is not applicable here) ---
    nb_clf = GaussianNB()
    nb_clf.fit(X_train, y_train)
    y_pred_nb = nb_clf.predict(X_test)
    results["NaiveBayes"] = evaluate(y_test, y_pred_nb, label_names, f"{tag} - NaiveBayes")

    return results, {"SVM": svm_clf, "NaiveBayes": nb_clf}
""")

# ---------------------------------------------------------------------------
md("## 7. Train/Test Split & Label Encoding")
code("""\
le = LabelEncoder()
y = le.fit_transform(df["category"])
label_names = list(le.classes_)

tokens_train, tokens_test, y_train, y_test = train_test_split(
    df["tokens"].tolist(), y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

all_results = {}
trained_artifacts = {}

print(f"Train size: {len(tokens_train)}, Test size: {len(tokens_test)}")
print(f"Categories: {label_names}")
""")

# ---------------------------------------------------------------------------
md("## 8. Word2Vec — Train, Vectorize, Classify, Evaluate")
code("""\
w2v_model = train_word2vec(tokens_train)
X_train_w2v = build_feature_matrix(tokens_train, "word2vec", w2v_model)
X_test_w2v = build_feature_matrix(tokens_test, "word2vec", w2v_model)

res_w2v, clfs_w2v = train_and_evaluate(
    X_train_w2v, X_test_w2v, y_train, y_test, label_names, "Word2Vec"
)
all_results["Word2Vec"] = res_w2v
trained_artifacts["Word2Vec"] = {"embedding_model": w2v_model, "classifiers": clfs_w2v}
""")

# ---------------------------------------------------------------------------
md("## 9. FastText — Train, Vectorize, Classify, Evaluate")
code("""\
ft_model = train_fasttext(tokens_train)
X_train_ft = build_feature_matrix(tokens_train, "fasttext", ft_model)
X_test_ft = build_feature_matrix(tokens_test, "fasttext", ft_model)

res_ft, clfs_ft = train_and_evaluate(
    X_train_ft, X_test_ft, y_train, y_test, label_names, "FastText"
)
all_results["FastText"] = res_ft
trained_artifacts["FastText"] = {"embedding_model": ft_model, "classifiers": clfs_ft}
""")

# ---------------------------------------------------------------------------
md("## 10. GloVe — Load Pretrained Vectors, Vectorize, Classify, Evaluate")
code("""\
try:
    glove_dict = load_glove(GLOVE_PATH)
    X_train_glove = build_feature_matrix(tokens_train, "glove", glove_dict)
    X_test_glove = build_feature_matrix(tokens_test, "glove", glove_dict)

    res_glove, clfs_glove = train_and_evaluate(
        X_train_glove, X_test_glove, y_train, y_test, label_names, "GloVe"
    )
    all_results["GloVe"] = res_glove
    trained_artifacts["GloVe"] = {"embedding_model": glove_dict, "classifiers": clfs_glove}
except FileNotFoundError as e:
    print(f"[Skipping GloVe] {e}")
""")

# ---------------------------------------------------------------------------
md("## 11. Compare Word2Vec vs GloVe vs FastText")
code("""\
rows = []
for embed_name, res in all_results.items():
    for clf_name, metrics in res.items():
        rows.append(
            {
                "Embedding": embed_name,
                "Classifier": clf_name,
                "Accuracy": metrics["accuracy"],
                "Precision": metrics["precision"],
                "Recall": metrics["recall"],
                "F1-score": metrics["f1"],
            }
        )
summary_df = pd.DataFrame(rows).sort_values("F1-score", ascending=False)
summary_df.to_csv("embedding_comparison_summary.csv", index=False)
summary_df
""")

# ---------------------------------------------------------------------------
md("## 12. Save the Best Model with Joblib")
code("""\
best_row = summary_df.iloc[0]
best_embed_name = best_row["Embedding"]
best_clf_name = best_row["Classifier"]
print(
    f"Best combination: {best_embed_name} + {best_clf_name} "
    f"(F1-score = {best_row['F1-score']:.4f})"
)

best_embedding_model = trained_artifacts[best_embed_name]["embedding_model"]
best_classifier = trained_artifacts[best_embed_name]["classifiers"][best_clf_name]

bundle = {
    "embedding_type": best_embed_name,
    "embedding_model": best_embedding_model,  # gensim model OR glove dict
    "classifier": best_classifier,
    "label_encoder": le,
    "vector_size": VECTOR_SIZE,
}
model_path = os.path.join(MODEL_OUT_DIR, "best_bbc_news_classifier.joblib")
joblib.dump(bundle, model_path)
print(f"Saved best model bundle to: {model_path}")
""")

# ---------------------------------------------------------------------------
md("## 13. Reload the Saved Model & Classify a New Article")
code("""\
def classify_new_article(text, bundle):
    \"\"\"Classify a brand-new news article string using a saved model bundle.\"\"\"
    tokens = tokenize(text)
    embed_type = bundle["embedding_type"]
    embed_key = embed_type.lower()
    vector_size = bundle["vector_size"]

    if embed_key == "glove":
        vec = document_vector_glove(tokens, bundle["embedding_model"], vector_size)
    else:
        vec = document_vector_gensim(tokens, bundle["embedding_model"], vector_size)

    vec = vec.reshape(1, -1)
    pred_encoded = bundle["classifier"].predict(vec)
    pred_label = bundle["label_encoder"].inverse_transform(pred_encoded)
    return pred_label[0]


loaded_bundle = joblib.load(model_path)

sample_article = (
    "The stock market rallied today as technology shares surged following "
    "strong earnings reports from major companies, boosting investor confidence."
)
predicted_category = classify_new_article(sample_article, loaded_bundle)

print(f"New article: {sample_article!r}")
print(f"Predicted category: {predicted_category}")
""")

nb["cells"] = cells

with open("/home/claude/bbc_embedding_lab.ipynb", "w") as f:
    nbf.write(nb, f)

print("Notebook written.")